In [ ]:
pip install openpyxl

In [ ]:
pip install pandas

In [ ]:
pip install numpy

In [ ]:
pip install plotly

In [ ]:
pip install matplotlib

In [ ]:
pip install seaborn

In [ ]:
pip install dash

In [ ]:
import plotly.graph_objects as go

In [ ]:
# Import standard libraries
import os
from contextlib import redirect_stdout

import sys
# append coeqwal packages to path
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
# Import custom libraries
# Note: on my computer the next import doesn't work the first time I call it, why? If I re-run the cell, then it is ok. MUST DEBUG
from coeqwalpackage.metrics import *
import cqwlutils as cu
import plotting as pu
import re
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

In [ ]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'
ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, DVDssMin, DVDssMax, SVDssMin, SVDssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

In [ ]:
# df, dss_names = read_in_df(ConvertDataOutPath,DVDssNamesOutPath)
df, dss_names = read_in_parquet_df(ConvertDataOutPath,DVDssNamesOutPath)

In [ ]:
df = add_water_year_column(df)

In [ ]:
df.columns = ['_'.join(map(str, col)) for col in df.columns]

print(df.columns.tolist()[:20]) 

In [ ]:
original_columns = df.columns.tolist()

s_numbers = set()
for col in original_columns:
    matches = re.findall(r's\d{4,}', col)  
    s_numbers.update(matches) 

# Convert to a sorted list
s_numbers_list = sorted(s_numbers)

print(s_numbers_list)

### Time Series
Shows how the selected variable changes over time for each scenario.

### Monthly-of-Year
Displays the monthly average for a selected year.

### Single Exceedance
Shows the probability that a value will be equaled or exceeded.

### Annual Exceedance
Shows how often the selected month's total value exceeds a given threshold across all years. Each year’s data for the chosen month is summed (e.g., total flow in April each year), and the annual values are ranked from highest to lowest. From these ranks, exceedance probabilities are calculated to show how frequently high values occur.

### Month-of-Year Avg
Averages each calendar month across all years, optionally filtered by Water Year Type. Water Year Types classify each year based on how wet or dry it was. The scale ranges from 1 (wettest) to 5 (driest)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
from dash.exceptions import PreventUpdate
from dash import callback_context

# --- Data Paths ---
DATA_PATH = "Data/ConvertDataFrom_trend_report_variables_v5.csv"
VARIABLE_DESC_PATH = "Data/trend_report_variables_v5.csv"
SCENARIO_DESC_PATH = "Data/coeqwal_cs3_scenario_listing_v5.csv"
VARIABLE_GROUPS_PATH = "Data/variable_groupings.csv"
SCENARIO_GROUPS_PATH = "Data/scenario_groupings.csv"

# --- Data Loading Functions ---
def read_calsim_data(path):
    df_head = pd.read_csv(path, header=None, nrows=10)
    header_rows = 0
    for i in range(len(df_head)):
        if isinstance(df_head.iloc[i, 0], str) and "Units" in str(df_head.iloc[i, 0]):
            header_rows = i
            break
    df_local = pd.read_csv(
        path,
        header=list(range(header_rows + 1)),
        parse_dates=[0],
        index_col=0,
        low_memory=False
    )
    if isinstance(df_local.columns, pd.MultiIndex):
        def flatten_col(col):
            parts = [str(x) for x in col if pd.notna(x) and str(x) != "nan"]
            return "_".join(parts)
        df_local.columns = [flatten_col(c) for c in df_local.columns]
    df_local.index = pd.to_datetime(df_local.index)
    return df_local

def read_csv_flexible(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except Exception:
        return pd.read_csv(path, encoding="latin1")

# --- Initial Data Loading and Metadata Processing ---
df = read_calsim_data(DATA_PATH)
df_var = read_csv_flexible(VARIABLE_DESC_PATH)
df_scen = read_csv_flexible(SCENARIO_DESC_PATH)
var_groups_raw = read_csv_flexible(VARIABLE_GROUPS_PATH)
scen_groups_raw = read_csv_flexible(SCENARIO_GROUPS_PATH)

original_columns = list(map(str, df.columns))

pat_var = re.compile(r"CALSIM_(.*?)_s\d{4}", re.IGNORECASE)
pat_s = re.compile(r"s\d{4,}", re.IGNORECASE)

records = []
for col in original_columns:
    m = pat_var.search(col)
    var = m.group(1) if m else None
    s_list = pat_s.findall(col)
    scen = s_list[0] if s_list else None
    unit = col.split("_")[-1] if "_" in col else ""
    records.append({"column": col, "variable": var, "scenario": scen, "unit": unit})

meta = pd.DataFrame(records)
scenarios = sorted({r["scenario"] for r in records if r["scenario"]})

variable_unit_pairs = []
for var, sub in meta.groupby("variable", dropna=True):
    units = {str(u).upper() for u in sub["unit"] if pd.notna(u)}
    for u in units:
        variable_unit_pairs.append((var, u))

variables = sorted([f"{v}__{u}" for v, u in variable_unit_pairs])

variable_labels = [
    {"label": f"{v} ({u})", "value": f"{v}__{u}"}
    for v, u in variable_unit_pairs
]

# --- Water Year and Month Definitions ---
def add_water_year_column(df_in):
    df_copy = df_in.copy().sort_index()
    df_copy["Date"] = pd.to_datetime(df_copy.index)
    df_copy["Year"] = df_copy["Date"].dt.year
    df_copy["Month"] = df_copy["Date"].dt.month
    df_copy["WaterYear"] = np.where(df_copy["Month"] >= 10, df_copy["Year"] + 1, df_copy["Year"])
    return df_copy.drop(["Date", "Year", "Month"], axis=1)

water_year_df = add_water_year_column(df)

water_month_order = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
month_name_map = {
    1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
    7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"
}
water_month_labels = [month_name_map[m] for m in water_month_order]

years_in_data = sorted(water_year_df["WaterYear"].unique())

drought_years = {
    1924, 1925, 1926, 1929, 1930, 1931, 1932, 1933, 1934, 1939,
    1944, 1945, 1947, 1948, 1949, 1950, 1955, 1960, 1961, 1962, 1964,
    1976, 1977, 1979, 1981, 1987, 1988, 1989, 1990, 1991, 1992, 1994,
    2001, 2008, 2009, 2013, 2014, 2015, 2020, 2021
}

year_options = [
    {
        "label": f"{y} {'(Historical Drought Year)' if y in drought_years else ''}",
        "value": y
    }
    for y in years_in_data
]

water_year_type_options = [{"label": str(i), "value": i} for i in range(1, 6)]

month_options = [
    {"label": month_name_map[m], "value": m}
    for m in water_month_order
]

# --- Plot Type Descriptions ---
plot_type_descriptions = {
    "time_series": "Shows how the selected variable changes over time for each scenario.",
    "monthly": "Displays the monthly average across selected water year(s).",
    "single_exceedance": "Shows the probability that a value will be equaled or exceeded.",
    "annual_exceedance": "Shows how often the selected month's total value exceeds a given threshold across all water years.",
    "month_of_year_avg": "Averages each calendar month across all water years, optionally filtered by Water Year Type."
}

# --- Helper Functions ---
def find_col(df_in, var_unit, scenario):
    var, unit = var_unit.split("__")
    suffix = f"_{unit.lower()}"
    matches = [
        c for c in df_in.columns
        if var in str(c)
        and scenario in str(c)
        and str(c).lower().endswith(suffix)
    ]
    return matches[0] if matches else None

def find_wyt_col(df_in, scenario):
    matches = [c for c in df_in.columns if f"CALSIM_WYT_SAC__{scenario}" in str(c) and "WATERYEARTYPE" in str(c)]
    return matches[0] if matches else None

def get_colors(scenarios_list):
    base = ["red", "blue", "green", "orange", "purple", "brown", "cyan", "magenta", "gray", "black"]
    return {s: base[i % len(base)] for i, s in enumerate(scenarios_list)}

def get_line_styles():
    return ["solid", "dash", "dot", "dashdot", "longdash", "longdashdot"]

def filter_by_wyt_annual(df_col, scenario, wyt_list, month=5):
    if not wyt_list:
        return df_col
    wyt_col = find_wyt_col(df, scenario)
    if wyt_col is None:
        return df_col
    working_df = df[[wyt_col]].copy()
    working_df["WaterYear"] = water_year_df["WaterYear"]
    working_df["Month"] = df.index.month
    filtered = working_df[working_df["Month"] == month].groupby("WaterYear").first()
    selected_years = filtered[filtered[wyt_col].isin(wyt_list)].index
    out = df_col.copy()
    out["WaterYear"] = water_year_df["WaterYear"]
    out = out[out["WaterYear"].isin(selected_years)]
    return out.drop(columns="WaterYear")

def normalize_scenario_token(tok):
    if tok is None:
        return None
    s = str(tok).strip()
    if not s:
        return None
    m = re.match(r"^s(\d+)$", s, flags=re.IGNORECASE)
    if m:
        n = int(m.group(1))
        return f"s{n:04d}"
    if re.match(r"^\d+$", s):
        n = int(s)
        return f"s{n:04d}"
    m2 = re.search(r"(s\d+)", s, flags=re.IGNORECASE)
    if m2:
        return normalize_scenario_token(m2.group(1))
    return None

def parse_unit_from_group_name(text):
    m = re.search(r"\(([^)]+)\)", str(text) if text is not None else "")
    return m.group(1).strip().upper() if m else None

def split_tokens(text):
    if text is None:
        return []
    s = str(text)
    parts = re.split(r"[,\n;|]+", s)
    return [p.strip() for p in parts if p and p.strip()]

def clean_var_token(token):
    if token is None:
        return None
    t = str(token).strip()
    if not t:
        return None
    t = re.sub(r"\s*\([^)]*\)\s*$", "", t).strip()
    return t if t else None

def unit_in_token(token):
    if token is None:
        return None
    m = re.search(r"\(([^)]+)\)\s*$", str(token).strip())
    return m.group(1).strip().upper() if m else None

def build_variable_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_vars = colmap.get("variables")
    if c_group is None or c_vars is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        unit = parse_unit_from_group_name(name)
        vars_raw = row.get(c_vars, "")
        tokens = split_tokens(vars_raw)
        cleaned = []
        for tok in tokens:
            v = clean_var_token(tok)
            if v:
                cleaned.append(tok.strip())
        groups[name] = {"description": desc, "unit": unit, "variables": cleaned}
    return groups

def build_scenario_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_sc = colmap.get("scenarios")
    if c_group is None or c_sc is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        sc_raw = row.get(c_sc, "")
        tokens = split_tokens(sc_raw)
        norm = []
        for tok in tokens:
            ns = normalize_scenario_token(tok)
            if ns:
                norm.append(ns)
        groups[name] = {"description": desc, "scenarios": norm}
    return groups

var_groups_store_init = build_variable_groups(var_groups_raw)
scen_groups_store_init = build_scenario_groups(scen_groups_raw)

available_varunit_values = {f"{v}__{u.upper()}" for v, u in variable_unit_pairs}

def group_options(store):
    return [{"label": k, "value": k} for k in sorted(store.keys())]

# --- Dash App Initialization ---
app = dash.Dash(__name__)
server = app.server

# --- Static Assets Setup ---
current_dir = os.getcwd()
assets_folder = os.path.join(current_dir, 'assets')
if not os.path.exists(assets_folder):
    try:
        os.makedirs(assets_folder)
        print(f"Created assets folder at: {assets_folder}")
    except Exception as e:
        print(f"Could not create assets folder: {e}")

logo_path = os.path.join(assets_folder, 'coeqwal_logo_outlines.png')
logo_exists = os.path.exists(logo_path)

# --- App Layout with Independent Controls for Each Plot ---
app.layout = html.Div(
    style={"fontFamily": "Inter, Arial, sans-serif", "padding": "30px"},
    children=[
        dcc.Store(id="var-groups-store", data=var_groups_store_init),
        dcc.Store(id="scen-groups-store", data=scen_groups_store_init),

        # Global controls (affect all plots)
        html.Img(
            src=app.get_asset_url('coeqwal_logo_outlines.png') if logo_exists else '',
            style={
                'height': '80px', 'display': 'block' if logo_exists else 'none', 
                'marginLeft': 'auto', 'marginRight': 'auto', 'marginBottom': '20px'
            }
        ),
        
        html.H1(
            "Water Data Dashboard",
            style={
                "textAlign": "center", "fontWeight": "800", "fontSize": "30px",
                "marginBottom": "40px", "color": "#135773"
            },
        ),

        # Global Selection Panel (affects all plots)
        html.Div(
            style={
                "maxWidth": "900px", "margin": "0 auto", "backgroundColor": "white",
                "padding": "30px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"
            },
            children=[
                html.H3("Global Selections (applies to all plots)", style={"color": "#135773", "marginBottom": "20px"}),
                
                html.Div(style={"display": "flex", "gap": "30px", "marginBottom": "25px"},
                    children=[
                        html.Div(style={"flex": "1"},
                            children=[
                                html.Label(
                                    "Variable Groups",
                                    style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}
                                ),
                                dcc.Dropdown(
                                    id="var-group-dropdown",
                                    options=group_options(var_groups_store_init),
                                    placeholder="Choose variable group..."
                                ),
                                html.H5(
                                    id="var-group-desc",
                                    children="Variable Group Represents:",
                                    style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"}
                                ),
                                html.Button("Add Variable Group to Selection", id="add-var-group-btn", n_clicks=0, style={"marginTop": "8px"}),
                            ]
                        ),
                        html.Div(
                            style={"flex": "1"},
                            children=[
                                html.Label(
                                    "Scenario Groups",
                                    style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}
                                ),
                                dcc.Dropdown(
                                    id="scen-group-dropdown",
                                    options=group_options(scen_groups_store_init),
                                    placeholder="Choose scenario group..."
                                ),
                                html.H5(
                                    id="scen-group-desc",
                                    children="Scenario Group Represents:",
                                    style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"}
                                ),
                                html.Button("Add Scenario Group to Selection", id="add-scen-group-btn", n_clicks=0, style={"marginTop": "8px"}),
                            ]
                        )
                    ]
                ),

                html.Div(style={"marginBottom": "25px"},
                    children=[
                        html.Label("Select Variables (must have same unit)", style={"fontWeight": "600", "display": "block"}),
                        dcc.Dropdown(
                            id="variable-dropdown",
                            options=variable_labels,
                            value=[],
                            multi=True
                        ),
                        html.Div(id="variable-description-output", style={"marginTop": "10px", "fontStyle": "italic", "color": "#555"}),
                    ]
                ),

                html.Div(style={"marginBottom": "25px"},
                    children=[
                        html.Label("Select Scenarios", style={"fontWeight": "600", "display": "block"}),
                        dcc.Dropdown(
                            id="scenario-dropdown",
                            options=[{"label": s, "value": s} for s in scenarios],
                            value=[],
                            multi=True
                        ),
                        html.Div(id="scenario-description-output", style={"marginTop": "10px", "fontStyle": "italic", "color": "#555"}),
                    ]
                ),
            ]
        ),

        # Individual Plot Sections with Their Own Controls
        html.Div(style={"marginTop": "40px"},
            children=[
                # Time Series Plot (no additional controls needed)
                html.Div(style={"backgroundColor": "white", "padding": "20px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"},
                    children=[
                        html.H2(id="title-time-series", style={"textAlign": "center", "marginBottom": "10px", "color": "#135773"}),
                        html.Div(plot_type_descriptions["time_series"], style={"textAlign": "center", "fontStyle": "italic", "marginBottom": "20px", "color": "#555"}),
                        dcc.Graph(id="plot-time-series"),
                    ]
                ),

                # Plot 2: Monthly Average for Multiple Selected Years
                html.Div(style={"backgroundColor": "white", "padding": "20px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"},
                    children=[
                        html.H2(id="title-monthly", style={"textAlign": "center", "marginBottom": "10px", "color": "#135773"}),
                        html.Div("Averages each month across multiple selected water years (combines data from all selected years)", 
                                style={"textAlign": "center", "fontStyle": "italic", "marginBottom": "20px", "color": "#555"}),
                        
                        # Independent controls for this plot
                        html.Div(style={"display": "flex", "gap": "20px", "marginBottom": "20px", "justifyContent": "center"},
                            children=[
                                html.Div(style={"width": "300px"},
                                    children=[
                                        html.Label("Select Multiple Water Years", style={"fontWeight": "600", "display": "block"}),
                                        dcc.Dropdown(id="year-dropdown-monthly", options=year_options, value=[], multi=True, placeholder="Choose years...")
                                    ]
                                ),
                            ]
                        ),
                        dcc.Graph(id="plot-monthly"),
                    ]
                ),

                # Plot 3: Month-of-Year Average for Selected Water Year Type
                html.Div(style={"backgroundColor": "white", "padding": "20px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"},
                    children=[
                        html.H2(id="title-moy", style={"textAlign": "center", "marginBottom": "10px", "color": "#135773"}),
                        html.Div("Averages each month across all years of selected Water Year Types.", 
                                style={"textAlign": "center", "fontStyle": "italic", "marginBottom": "20px", "color": "#555"}),
                        
                        # Independent controls for this plot
                        html.Div(style={"display": "flex", "gap": "20px", "marginBottom": "20px", "justifyContent": "center"},
                            children=[
                                html.Div(style={"width": "300px"},
                                    children=[
                                        html.Label("Select Water Year Type (1-5)", style={"fontWeight": "600", "display": "block"}),
                                        dcc.Dropdown(id="wyt-dropdown-moy", options=water_year_type_options, value=[], multi=True, placeholder="Choose WYT..."),
                                        html.Div("1 = wettest, 5 = driest", style={"fontSize": "12px", "marginTop": "4px", "color": "#555"})
                                    ]
                                ),
                            ]
                        ),
                        dcc.Graph(id="plot-moy"),
                    ]
                ),

                # Single Exceedance Plot (no additional controls needed)
                html.Div(style={"backgroundColor": "white", "padding": "20px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"},
                    children=[
                        html.H2(id="title-single-exceedance", style={"textAlign": "center", "marginBottom": "10px", "color": "#135773"}),
                        html.Div(plot_type_descriptions["single_exceedance"], style={"textAlign": "center", "fontStyle": "italic", "marginBottom": "20px", "color": "#555"}),
                        dcc.Graph(id="plot-single-exceedance"),
                    ]
                ),

                # Annual Exceedance Plot
                html.Div(style={"backgroundColor": "white", "padding": "20px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)", "marginBottom": "30px"},
                    children=[
                        html.H2(id="title-annual-exceedance", style={"textAlign": "center", "marginBottom": "10px", "color": "#135773"}),
                        html.Div(plot_type_descriptions["annual_exceedance"], style={"textAlign": "center", "fontStyle": "italic", "marginBottom": "20px", "color": "#555"}),
                        
                        # Independent controls for this plot
                        html.Div(style={"display": "flex", "gap": "20px", "marginBottom": "20px", "justifyContent": "center"},
                            children=[
                                html.Div(style={"width": "300px"},
                                    children=[
                                        html.Label("Select Month for Annual Exceedance", style={"fontWeight": "600", "display": "block"}),
                                        dcc.Dropdown(id="month-dropdown-annual", options=month_options, value=10, placeholder="Choose month...")
                                    ]
                                ),
                            ]
                        ),
                        dcc.Graph(id="plot-annual-exceedance"),
                    ]
                ),
            ]
        )
    ]
)

# --- Callbacks for Group Descriptions (unchanged) ---
@app.callback(
    Output("var-group-desc", "children"),
    [Input("var-group-dropdown", "value"),
     Input("var-groups-store", "data")]
)
def show_var_group_desc(group_name, store):
    if not group_name or not store or group_name not in store:
        return ""
    desc = store[group_name].get("description", "")
    unit = store[group_name].get("unit")
    return f"{desc} (Unit: {unit})" if unit else desc

@app.callback(
    Output("scen-group-desc", "children"),
    [Input("scen-group-dropdown", "value"),
     Input("scen-groups-store", "data")]
)
def show_scen_group_desc(group_name, store):
    if not group_name or not store or group_name not in store:
        return ""
    desc = store[group_name].get("description", "")
    scs = store[group_name].get("scenarios", []) or []
    scs_avail = [s for s in scs if s in scenarios]
    return f"{desc}"

# --- Callbacks for Adding Groups to Selection (unchanged) ---
@app.callback(
    Output("variable-dropdown", "value"),
    Input("add-var-group-btn", "n_clicks"),
    State("var-group-dropdown", "value"),
    State("var-groups-store", "data"),
    State("variable-dropdown", "value"),
    prevent_initial_call=True
)
def add_var_group_to_selection(_, group_name, store, current_selected):
    if not group_name or not store or group_name not in store:
        raise PreventUpdate
    current_selected = list(current_selected or [])
    group_unit = (store[group_name].get("unit") or "").upper() or None
    vars_list = store[group_name].get("variables", []) or []
    add_vals = []
    for raw_tok in vars_list:
        tok_unit = unit_in_token(raw_tok)
        vname = clean_var_token(raw_tok)
        if not vname:
            continue
        use_unit = tok_unit or group_unit
        if not use_unit:
            continue
        cand = f"{vname}__{use_unit}"
        if cand in available_varunit_values:
            add_vals.append(cand)
    merged = list(dict.fromkeys(current_selected + add_vals))
    return merged

@app.callback(
    Output("scenario-dropdown", "value"),
    Input("add-scen-group-btn", "n_clicks"),
    State("scen-group-dropdown", "value"),
    State("scen-groups-store", "data"),
    State("scenario-dropdown", "value"),
    prevent_initial_call=True
)
def add_scen_group_to_selection(_, group_name, store, current_selected):
    if not group_name or not store or group_name not in store:
        raise PreventUpdate
    current_selected = list(current_selected or [])
    group_scen = store[group_name].get("scenarios", []) or []
    group_scen_avail = [s for s in group_scen if s in scenarios]
    merged = list(dict.fromkeys(current_selected + group_scen_avail))
    return merged

# --- Callback for Variable Descriptions (unchanged) ---
@app.callback(
    Output("variable-description-output", "children"),
    Input("variable-dropdown", "value"),
)
def display_variable_descriptions(vars_selected):
    if not vars_selected:
        return "Select Variables"
    
    var_names = [v.split("__")[0] for v in vars_selected]
    
    col_var = next((c for c in df_var.columns if "Variable" in c or "Unnamed: 3" in str(c)), df_var.columns[0])
    col_desc = next((c for c in df_var.columns if "Description" in c or "Unnamed: 10" in str(c)), df_var.columns[1] if len(df_var.columns) > 1 else df_var.columns[0])
    
    df_var_clean = df_var.copy()
    df_var_clean[col_var] = df_var_clean[col_var].astype(str).str.strip()
    
    variable_defs = (
        df_var_clean[df_var_clean[col_var].isin(var_names)]
        .set_index(col_var)[col_desc]
        .to_dict()
    )

    return [
        html.Div([
            html.B(f"{var}: "),
            html.Span(desc)
        ]) for var, desc in variable_defs.items()
    ]

# --- Callback for Scenario Descriptions (unchanged) ---
@app.callback(
    Output("scenario-description-output", "children"),
    Input("scenario-dropdown", "value"),
)
def display_scenario_descriptions(scen_list):
    if not scen_list:
        return "Select Scenarios"
    
    col_idx = next((c for c in df_scen.columns if "Index" in c), df_scen.columns[0])
    col_desc = next((c for c in df_scen.columns if "ShortDescription" in c or "Description" in c), df_scen.columns[1] if len(df_scen.columns) > 1 else df_scen.columns[0])
    
    scenario_defs = (
        df_scen[df_scen[col_idx].isin(scen_list)]
        .set_index(col_idx)[col_desc]
        .to_dict()
    )

    return [
        html.Div([
            html.B(f"{scen}: "),
            html.Span(desc)
        ]) for scen, desc in scenario_defs.items()
    ]

# --- Individual Plot Callbacks ---

# Time Series Plot Callback
@app.callback(
    [Output("plot-time-series", "figure"),
     Output("title-time-series", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value")]
)
def update_time_series_plot(vars_selected, scenarios_selected):
    def empty_figure(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scenarios_selected:
        return empty_figure("Time Series Plot"), "Time Series Plot"

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units. Please select variables with the same unit.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, "Time Series Plot"

    unit_for_axis = selected_units.pop()
    colors = get_colors(scenarios_selected)
    line_styles = get_line_styles()
    
    fig = go.Figure()
    var_names = [v.split("__")[0] for v in vars_selected]
    var_names_str = ", ".join(var_names)

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]

        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue

            fig.add_trace(go.Scatter(
                x=df.index,
                y=df[col],
                mode="lines",
                name=f"{s} - {var} ({unit})",
                line=dict(color=colors[s], dash=style),
            ))

    fig.update_layout(
        xaxis_title="Date", 
        yaxis_title=f"Value ({unit_for_axis})"
    )
    title = f"Time Series Plot: {var_names_str} ({unit_for_axis})"
    
    return fig, title

# Plot 2: Monthly Average for Multiple Selected Years
@app.callback(
    [Output("plot-monthly", "figure"),
     Output("title-monthly", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value"),
     Input("year-dropdown-monthly", "value")]
)
def update_monthly_plot(vars_selected, scenarios_selected, years_selected):
    def empty_figure(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scenarios_selected:
        return empty_figure("Monthly Average (Multiple Years)"), "Monthly Average (Multiple Years)"

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, "Monthly Average (Multiple Years)"

    unit_for_axis = selected_units.pop()
    colors = get_colors(scenarios_selected)
    line_styles = get_line_styles()
    years_selected = list(years_selected or [])
    
    fig = go.Figure()
    var_names = [v.split("__")[0] for v in vars_selected]
    var_names_str = ", ".join(var_names)

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]

        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue

            df_copy = df[[col]].copy()
            df_copy["WaterYear"] = water_year_df["WaterYear"]
            df_copy["Month"] = df_copy.index.month
            df_copy["WaterMonth"] = ((df_copy["Month"] - 10) % 12) + 1
            
            if years_selected:
                df_sel = df_copy[df_copy["WaterYear"].isin(years_selected)]
            else:
                df_sel = df_copy.iloc[0:0]
                
            monthly_avg = df_sel.groupby("WaterMonth")[col].mean()
            monthly_avg = monthly_avg.reindex(range(1, 13))
            
            fig.add_trace(go.Scatter(
                x=monthly_avg.index,
                y=monthly_avg.values,
                mode="lines+markers",
                name=f"{s} - {var} ({unit})",
                line=dict(color=colors[s], dash=style)
            ))

    fig.update_layout(
        xaxis_title="Month of Water Year (Oct–Sep)",
        yaxis_title=f"Value ({unit_for_axis})"
    )
    fig.update_xaxes(
        tickmode="array", tickvals=list(range(1, 13)), ticktext=water_month_labels
    )
    
    year_text = f" (Years: {', '.join(map(str, years_selected))})" if years_selected else " (No years selected)"
    title = f"Monthly Average (Multiple Years): {var_names_str} ({unit_for_axis}){year_text}"
    
    return fig, title

# Plot 3: Month-of-Year Average for Selected Water Year Type
@app.callback(
    [Output("plot-moy", "figure"),
     Output("title-moy", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value"),
     Input("wyt-dropdown-moy", "value")]
)
def update_moy_wyt_plot(vars_selected, scenarios_selected, wyt_selected):
    def empty_figure(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scenarios_selected:
        return empty_figure("Water Year Type Average"), "Water Year Type Average"

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, "Water Year Type Average"

    unit_for_axis = selected_units.pop()
    colors = get_colors(scenarios_selected)
    line_styles = get_line_styles()
    wyt_selected = list(wyt_selected or [])
    
    fig = go.Figure()
    var_names = [v.split("__")[0] for v in vars_selected]
    var_names_str = ", ".join(var_names)

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]

        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue

            df_sel_moy = water_year_df[[col]].copy()
            df_sel_moy = filter_by_wyt_annual(df_sel_moy, s, wyt_selected, month=5)
            df_sel_moy["Month"] = df_sel_moy.index.month
            df_sel_moy["WaterMonth"] = ((df_sel_moy["Month"] - 10) % 12) + 1
            
            monthly_avg_moy = df_sel_moy.groupby("WaterMonth")[col].mean()
            monthly_avg_moy = monthly_avg_moy.reindex(range(1, 13))
            
            fig.add_trace(go.Scatter(
                x=monthly_avg_moy.index,
                y=monthly_avg_moy.values,
                mode="lines",
                name=f"{s} - {var} ({unit})",
                line=dict(color=colors[s], dash=style)
            ))

    fig.update_layout(
        xaxis_title="Month of Water Year (Oct–Sep)",
        yaxis_title=f"Value ({unit_for_axis})"
    )
    fig.update_xaxes(
        tickmode="array", tickvals=list(range(1, 13)), ticktext=water_month_labels
    )
    
    wyt_text = f" (WYT: {', '.join(map(str, wyt_selected))})" if wyt_selected else " (No WYT selected - all years)"
    title = f"Water Year Type Average: {var_names_str} ({unit_for_axis}){wyt_text}"
    
    return fig, title

# Single Exceedance Plot
@app.callback(
    [Output("plot-single-exceedance", "figure"),
     Output("title-single-exceedance", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value")]
)
def update_single_exceedance_plot(vars_selected, scenarios_selected):
    def empty_figure(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scenarios_selected:
        return empty_figure("Single Exceedance Plot"), "Single Exceedance Plot"

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, "Single Exceedance Plot"

    unit_for_axis = selected_units.pop()
    colors = get_colors(scenarios_selected)
    line_styles = get_line_styles()
    
    fig = go.Figure()
    var_names = [v.split("__")[0] for v in vars_selected]
    var_names_str = ", ".join(var_names)

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]

        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue

            series = df[col].dropna().sort_values(ascending=False)
            exceedance_probs = np.arange(1, len(series) + 1) / (len(series) + 1)
            
            fig.add_trace(go.Scatter(
                x=exceedance_probs,
                y=series.values,
                mode="lines",
                name=f"{s} - {var} ({unit})",
                line=dict(color=colors[s], dash=style)
            ))

    fig.update_layout(
        xaxis_title="Exceedance Probability", 
        yaxis_title=f"Value ({unit_for_axis})"
    )
    title = f"Single Exceedance Plot: {var_names_str} ({unit_for_axis})"
    
    return fig, title

# Annual Exceedance Plot
@app.callback(
    [Output("plot-annual-exceedance", "figure"),
     Output("title-annual-exceedance", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value"),
     Input("month-dropdown-annual", "value")]
)
def update_annual_exceedance_plot(vars_selected, scenarios_selected, selected_month):
    def empty_figure(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scenarios_selected:
        return empty_figure("Annual Exceedance Plot"), "Annual Exceedance Plot"

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, "Annual Exceedance Plot"

    unit_for_axis = selected_units.pop()
    colors = get_colors(scenarios_selected)
    line_styles = get_line_styles()
    
    fig = go.Figure()
    var_names = [v.split("__")[0] for v in vars_selected]
    var_names_str = ", ".join(var_names)
    
    selected_month = selected_month or 10  # Default to October if nothing selected

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]

        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue

            df_sel_ann = df[df.index.month == selected_month].copy()
            df_sel_ann["WaterYear"] = water_year_df.loc[df_sel_ann.index, "WaterYear"]
            annual_sum = df_sel_ann.groupby("WaterYear")[col].sum(min_count=1)
            sorted_vals = annual_sum.dropna().sort_values(ascending=False)
            exceed_probs = sorted_vals.rank(method="first", ascending=False) / (1 + len(sorted_vals))
            
            fig.add_trace(go.Scatter(
                x=exceed_probs,
                y=sorted_vals.values,
                mode="lines",
                name=f"{s} - {var} ({unit})",
                line=dict(color=colors[s], dash=style)
            ))

    fig.update_layout(
        xaxis_title="Exceedance Probability", 
        yaxis_title=f"Value ({unit_for_axis})"
    )
    
    month_name = month_name_map.get(selected_month, f"Month {selected_month}")
    title = f"Annual Exceedance Plot: {var_names_str} ({unit_for_axis}) - {month_name}"
    
    return fig, title

if __name__ == "__main__":
    app.run(debug=True, port=8060)

### Save data